# Fit transformations correctly

**Course:** C04, SQL and Data Preparation for AI · **Lesson:** [C04-M05-L03 Fit transformations correctly](https://learning.nextia-ai.com/courses/sql/m05/fit-transformations-correctly/)

**You will learn to** fit preprocessing on the train set only, keep it that way with a scikit-learn pipeline, and see how much leaky columns inflate a score.

**Time:** about 35 minutes. **Needs:** no account. The first cell installs pandas and scikit-learn.

**Tested:** 2026-10-06, Python 3.14.6 with pandas 3.0.6 and scikit-learn 1.9.1, in Jupyter (nbconvert) on macOS, from a fresh start (*Restart and run all*). Data: Larkfield's synthetic help-desk tickets ([dataset card](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/data/dataset.md)). The data is made up for this course; it has no real people.

## Setup

Run the next cell. It installs the two packages in the versions that the course uses, and imports them. It takes up to a minute the first time.

You should see `pandas 3.0.6, scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**) and run the cell again. If the installation fails, check your internet connection.

In [ ]:
%pip install --quiet --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1

import hashlib
import urllib.request
from pathlib import Path

import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print(f"pandas {pd.__version__}, scikit-learn {sklearn.__version__}")

## Helpers

The next cell defines the names of the columns and three small helpers. You can read them, but you do not need to change them. Run the cell. It prints nothing.

- `build_pipeline` makes the preprocessing and the model as one object, as in the lesson.
- `split_by_time` makes the same train, validation and test sets as `split.py`.
- `auc_on_valid` fits a pipeline on the train set only and returns its ROC AUC on the validation set.

In [ ]:
CATEGORIES = ["channel", "team", "segment", "region"]
NUMBERS = ["priority", "order_value", "word_count", "customer_tenure_days", "prior_tickets_90d", "created_hour"]
TARGET = "escalated_72h"


def build_pipeline(categories, numbers):
    """Preprocessing (one-hot, median imputation, scaling) and a logistic regression, as one pipeline."""
    number_steps = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)), ("scale", StandardScaler())])
    prepare = ColumnTransformer([("categories", OneHotEncoder(handle_unknown="ignore"), categories), ("numbers", number_steps, numbers)])
    return Pipeline([("prepare", prepare), ("model", LogisticRegression(max_iter=1000))])


def split_by_time(df):
    """The split of split.py: train before May, validation in May, test in June."""
    train = df[df["created_at"] < "2026-05-01"]
    valid = df[(df["created_at"] >= "2026-05-01") & (df["created_at"] < "2026-06-01")]
    test = df[df["created_at"] >= "2026-06-01"]
    return train, valid, test


def auc_on_valid(categories, numbers, train, valid):
    """Fit on train only, then return the ROC AUC on valid, rounded to 3 decimals."""
    pipeline = build_pipeline(categories, numbers).fit(train[categories + numbers], train[TARGET])
    scores = pipeline.predict_proba(valid[categories + numbers])[:, 1]
    return round(roc_auc_score(valid[TARGET], scores), 3)

## The data, briefly

The file `tickets_model.csv` is the modelling table from Lessons 1 and 2: one row per ticket, 10 features and the target `escalated_72h`. It also has two columns that a model must **not** use, `priority_now` and `first_reply_minutes`. They are here for the leakage experiments.

Run the next cell. It uses `tickets_model.csv` from the notebook's folder if it is there. If not, it downloads the file and checks its SHA-256 checksum, so you know that you have the expected version.

You should see `1369 rows, checksum OK`, then the size and the escalation rate of each part. Train has 927 rows and 0.117, validation 227 and 0.198, and test 215 and 0.163.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/data/tickets_model.csv"
DATA_SHA256 = "cd0dffc4e29ac09d538da02c6d62633779022575f20c0ab8efb72abaade70d3e"
DATA_FILE = Path("tickets_model.csv")

if not DATA_FILE.exists():
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)
checksum = hashlib.sha256(DATA_FILE.read_bytes()).hexdigest()
assert checksum == DATA_SHA256, "tickets_model.csv is not the expected file. Delete it, then run this cell again."

tickets = pd.read_csv(DATA_FILE, parse_dates=["created_at"])
print(len(tickets), "rows, checksum OK")

train, valid, test = split_by_time(tickets)
for name, part in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name:6} {len(part):4} rows, escalation rate {part[TARGET].mean():.3f}")

## The idea on a tiny example: fit, then transform

A preprocessing step has two parts:

- **fit** learns numbers from data. A median imputer learns the median. A scaler learns the mean and the standard deviation (SD).
- **transform** uses the learned numbers to change data. The imputer fills a missing value with the learned median. The scaler computes (value − mean) ÷ SD.

You fit on the train set only. Then you transform the train, validation and test sets with the **same** learned numbers. The test set must stay unseen until the end, like new tickets in real use.

Run the next cell. It fits on five train values and transforms two test values. You should see the median `50`, the mean `54` and the SD `26.53`, and the test values `-0.15` and `13.04`.

In [ ]:
tiny_train = pd.DataFrame({"order_value": [20.0, 40.0, None, 60.0, 100.0]})
tiny_test = pd.DataFrame({"order_value": [None, 400.0]})

imputer = SimpleImputer(strategy="median").fit(tiny_train)
scaler = StandardScaler().fit(imputer.transform(tiny_train))
print("median learned from train:", imputer.statistics_)
print("mean and SD learned from train:", scaler.mean_.round(2), scaler.scale_.round(2))
print("test after transform:", scaler.transform(imputer.transform(tiny_test)).round(2).ravel())

Check it by hand. The four known train values are 20, 40, 60 and 100, so the median is (40 + 60) ÷ 2 = 50. After the gap is filled, the mean is (20 + 40 + 50 + 60 + 100) ÷ 5 = 54. The missing test value becomes 50, and (50 − 54) ÷ 26.53 = −0.15. The value 400 becomes (400 − 54) ÷ 26.53 = 13.04: far above anything in train, and the transformed value says so.

Now the mistake: fit on the train **and** test values together. Run the next cell. You should see a median of `60`, a mean of `105.71` and an SD of `122.22`.

In [ ]:
tiny_both = pd.concat([tiny_train, tiny_test])
imputer_all = SimpleImputer(strategy="median").fit(tiny_both)
scaler_all = StandardScaler().fit(imputer_all.transform(tiny_both))
print("median:", imputer_all.statistics_, "mean and SD:", scaler_all.mean_.round(2), scaler_all.scale_.round(2))
print("train after transform:", scaler_all.transform(imputer_all.transform(tiny_train)).round(2).ravel())

One test value, 400, moved every learned number. The train values now look small and close together, because the test value stretched the scale. The preprocessing knows something about the test set before the test. On real data, the effect is usually smaller, as you see below. But the rule is simple: **fit on train, transform everything else.**

## The idea on the real data: a pipeline

A **pipeline** joins the preprocessing and the model into one object. `pipeline.fit(train)` fits every step on the train rows only. `pipeline.predict_proba(valid)` only transforms the validation rows with what it learned. So the rule is kept by the code, not by your memory.

Run the next cell. It fits the pipeline from the helpers on the train set and prints what it learned, then the score on the validation set. You should see a median of `86.21` for `order_value` and a ROC AUC of `0.673`.

**ROC AUC** in one sentence: the chance that the model scores a random escalated ticket higher than a random other ticket (0.5 is guessing, 1.0 is a perfect ranking). Course C05 explains it in full.

In [ ]:
pipeline = build_pipeline(CATEGORIES, NUMBERS).fit(train[CATEGORIES + NUMBERS], train[TARGET])
learned = pipeline.named_steps["prepare"].named_transformers_["numbers"]["impute"].statistics_
print("medians learned from train:", {name: float(value) for name, value in zip(NUMBERS, learned)})

honest_auc = auc_on_valid(CATEGORIES, NUMBERS, train, valid)
print("ROC AUC on valid, honest features:", honest_auc)

0.673 is a modest score. The model ranks escalated tickets above the others more often than guessing, but far from always. That is the honest starting point.

## Predict, then run: add `priority_now`

Tomás wants to add `priority_now`, the current priority. The help desk sets it to 1 when a ticket is escalated.

1. **Predict:** what ROC AUC do you expect with `priority_now` as an extra number? Write your prediction in the next cell, between the quotes.
2. Run the cell, and compare.

In [ ]:
my_prediction = ""  # for example "about 0.7" or "above 0.9"

with_priority_now = auc_on_valid(CATEGORIES, NUMBERS + ["priority_now"], train, valid)
print("your prediction:", my_prediction or "(none)")
print("ROC AUC on valid with priority_now:", with_priority_now)

You should see `0.964`, from 0.673. The column does not predict the escalation; it records it. Every escalated ticket has `priority_now` = 1, because the value is written **after** the escalation. When a new ticket arrives, `priority_now` is the same as `priority`, and the 0.964 disappears. This is **leakage**: information in the features that the model would not have at prediction time.

Run the next cell. It adds `first_reply_minutes` instead, the minutes until the first reply. It is not the target, but it is also written after creation. You should see `0.856`.

In [ ]:
with_first_reply = auc_on_valid(CATEGORIES, NUMBERS + ["first_reply_minutes"], train, valid)
print("ROC AUC on valid with first_reply_minutes:", with_first_reply)

Escalated tickets get a fast first reply from a senior agent, so the reply time is a trace of the escalation. A leak does not have to be the target itself to inflate the score.

## Change one thing: fit the preprocessing on all rows

Now change one thing in the honest pipeline: fit the **preprocessing** on all 1,369 rows (train, validation and test), and the model on train only. This is the mistake "fitting preprocessing on test data".

**Predict:** will the score go up a lot, a little, or not at all? Then run the next cell.

In [ ]:
features = CATEGORIES + NUMBERS
prepare_all = build_pipeline(CATEGORIES, NUMBERS).named_steps["prepare"].fit(tickets[features])
model = LogisticRegression(max_iter=1000).fit(prepare_all.transform(train[features]), train[TARGET])
scores = model.predict_proba(prepare_all.transform(valid[features]))[:, 1]

print("median prior_tickets_90d learned from all rows:", float(prepare_all.named_transformers_["numbers"]["impute"].statistics_[4]))
print("ROC AUC, preprocessing fit on all rows:", round(roc_auc_score(valid[TARGET], scores), 4))
print("ROC AUC, preprocessing fit on train only:", round(roc_auc_score(valid[TARGET], pipeline.predict_proba(valid[features])[:, 1]), 4))

You should see a median of `5.0` for `prior_tickets_90d` (it is `3.0` from train only), and two scores that differ only in the fourth decimal: `0.6726` and `0.6728`. Be honest about it: **on this data, fitting a median and a scaler on all rows changes the score very little.** A median and a mean carry little information about the target.

The mistake is still a mistake. The learned numbers came from the future, and the effect grows with small data, rare categories, or steps that learn from the target. The next cell shows a step that learns from the target: each customer's escalation rate, computed over **all** rows, including validation and test. You should see `0.767`.

In [ ]:
customer_rate_all = tickets.groupby("customer_id")[TARGET].mean()
leaky = tickets.assign(customer_rate_all=tickets["customer_id"].map(customer_rate_all))
leaky_train, leaky_valid, _ = split_by_time(leaky)
print("ROC AUC on valid with customer_rate_all:", auc_on_valid(CATEGORIES, NUMBERS + ["customer_rate_all"], leaky_train, leaky_valid))

The rate for a May ticket includes the May ticket's own target. The model reads part of the answer from the feature, and the score rises from 0.673 to 0.767. Leaky features and leaky aggregates are where the large inflation comes from.

| Features | ROC AUC on valid |
|---|---|
| Honest features | 0.673 |
| + `priority_now` | 0.964 |
| + `first_reply_minutes` | 0.856 |
| + customer escalation rate over all rows | 0.767 |
| Preprocessing fit on all rows | 0.6726 (honest: 0.6728) |

## Failure case: a category that train never saw

Larkfield starts to answer messages on social media. A new ticket arrives with the channel `social`. No train ticket has this channel, so the one-hot encoder never learned it. **One-hot encoding** turns a category into columns of 0 and 1, one column per category that the encoder learned.

Run the next cell. The first encoder uses the default setting and fails. The second one uses `handle_unknown="ignore"`. You should see a `ValueError` that names `social`, then a row of five zeros for `social` and a 1 in the first column for `chat`.

In [ ]:
new_ticket = pd.DataFrame({"channel": ["social"]})

strict = OneHotEncoder().fit(train[["channel"]])
try:
    strict.transform(new_ticket)
except ValueError as error:
    print("ValueError:", error)

tolerant = OneHotEncoder(handle_unknown="ignore").fit(train[["channel"]])
print(tolerant.get_feature_names_out())
print(tolerant.transform(new_ticket).toarray())
print(tolerant.transform(pd.DataFrame({"channel": ["chat"]})).toarray())

With the default, one new category stops every prediction: in real use, the routing service fails on the first social ticket. With `handle_unknown="ignore"`, the ticket gets zeros in all channel columns: "none of the channels I know". The model still gives a score from the other features.

**Fix:** use `OneHotEncoder(handle_unknown="ignore")`, as the pipeline does. Do not fix it by fitting the encoder on all rows. That works only for the categories in your file, not for the next new one. Also count unknown categories in new data, so you see when they become common.

## Check yourself

Each exercise ends with a cell of `assert` lines. If your answer is correct, the cell prints `Correct.` If not, it shows a message that says what to check.

**Exercise 1: the final test score.** Priya is done choosing. Now she fits the honest pipeline once on train **and** validation together, and scores the test set (June) once. In the next cell, fit `final_pipeline` on `train_and_valid` with `CATEGORIES + NUMBERS`, and put its ROC AUC on `test`, rounded to 3 decimals, in `test_auc`.

In [ ]:
train_and_valid = pd.concat([train, valid])
final_pipeline = build_pipeline(CATEGORIES, NUMBERS)
final_pipeline.fit(train_and_valid[CATEGORIES + NUMBERS], train_and_valid[TARGET])
test_scores = final_pipeline.predict_proba(test[CATEGORIES + NUMBERS])[:, 1]
test_auc = round(roc_auc_score(test[TARGET], test_scores), 3)
print(test_auc)

In [ ]:
assert test_auc is not ..., "Replace ... with your code."
assert hasattr(final_pipeline.named_steps["model"], "coef_"), "Fit final_pipeline before you score the test set."
assert final_pipeline.named_steps["prepare"].named_transformers_["numbers"]["impute"].n_features_in_ == len(NUMBERS), "Use CATEGORIES + NUMBERS only."
assert round(test_auc, 3) == 0.761, f"test_auc is {test_auc}. Fit on train_and_valid (1,154 rows), score on test (215 rows), and round to 3 decimals."
print("Correct.")

**Exercise 2: find the seeded leak.** A colleague offers three new columns. One of them leaks. Run the next cell first: it makes the three columns. Each one is a category.

- `weekday`: the day of the week when the ticket was created.
- `order_size`: `small`, `medium`, `large` or `none`, from `order_value`.
- `reply_speed`: `fast` or `slow`.

In [ ]:
seeded = tickets.assign(
    weekday=tickets["created_at"].dt.day_name(),
    order_size=pd.cut(tickets["order_value"], [0, 50, 150, float("inf")], labels=["small", "medium", "large"])
    .cat.add_categories("none").fillna("none").astype("str"),
    reply_speed=(tickets["first_reply_minutes"] < 30).map({True: "fast", False: "slow"}),
)
seeded_train, seeded_valid, _ = split_by_time(seeded)
print(seeded[["weekday", "order_size", "reply_speed"]].head(3))

In the next cell, add each column alone to the honest features, and compute the ROC AUC on validation with `auc_on_valid`. Put the three scores in the dictionary `scores_by_column`. Then set `leaky_column` to the name of the column that leaks.

Use the scores as a clue, then confirm with a question: **when is the value written?**

In [ ]:
scores_by_column = {}
for column in ["weekday", "order_size", "reply_speed"]:
    scores_by_column[column] = auc_on_valid(CATEGORIES + [column], NUMBERS, seeded_train, seeded_valid)
print(scores_by_column)

leaky_column = "reply_speed"

In [ ]:
assert all(isinstance(v, float) for v in scores_by_column.values()), "Each value in scores_by_column must be a ROC AUC from auc_on_valid."
assert leaky_column in scores_by_column, 'Set leaky_column to "weekday", "order_size" or "reply_speed".'
assert leaky_column == max(scores_by_column, key=scores_by_column.get), "Look again at the three scores: which column lifts the score far above 0.673?"
assert leaky_column == "reply_speed", "Ask when each value is written."
print("Correct.")

`reply_speed` comes from `first_reply_minutes`, which is written after the first reply, so it is not known when the ticket arrives. A new name and a coarser form do not remove a leak. `weekday` and `order_size` are known at creation, and they change the score very little.

In the lesson's module check, you write two or three sentences on how this leak inflated the evaluation. Write them in the next cell now, while the numbers are in front of you.

*Your explanation here (double-click to edit):*

## Recap

- **Fit** learns numbers (medians, means, SDs, the list of categories); **transform** uses them. Fit on the train set only, and transform the validation and test sets with the same numbers.
- A scikit-learn **pipeline** keeps that rule in the code: `fit` on train, `predict_proba` on everything else.
- `OneHotEncoder(handle_unknown="ignore")` turns a category it never saw, such as `social`, into zeros instead of an error.
- On this data, fitting the preprocessing on all rows changed the score only in the fourth decimal. The large inflation came from leaky columns. The honest score is 0.673. It becomes 0.964 with `priority_now`, 0.856 with `first_reply_minutes` and 0.767 with a customer rate over all rows.

Next: go back to the lesson for the module practice and the module check: [Fit transformations correctly](https://learning.nextia-ai.com/courses/sql/m05/fit-transformations-correctly/).